# Sentiment Prediction

### Import libraries

In [27]:
import re
import numpy as np

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model


In [ ]:
# Load IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {val:key for key, val in word_index.items()}

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


### Load model

In [ ]:
# Load pre-trained model with ReLU activation
model = load_model('/content/simple_rnn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
model.get_weights()

[array([[ 0.00840849,  0.01839163, -0.10059197, ..., -0.03018615,
         -0.014862  , -0.10741663],
        [-0.04309547, -0.07184956,  0.01066653, ..., -0.01382845,
         -0.00358578,  0.07717403],
        [-0.12154659,  0.02350025,  0.06068783, ..., -0.0273939 ,
          0.18342753, -0.13456506],
        ...,
        [ 0.00303566, -0.00470124,  0.08004456, ...,  0.07465896,
         -0.02566046,  0.12344575],
        [ 0.09115903, -0.03114885,  0.00145972, ...,  0.01601953,
          0.0025206 , -0.02725041],
        [-0.01597546, -0.08178243,  0.07834151, ...,  0.11287142,
         -0.12927455,  0.11041202]], dtype=float32),
 array([[-0.08167995,  0.13295966, -0.10084903, ...,  0.01727293,
         -0.12742156,  0.21340491],
        [-0.08762889,  0.08003973,  0.05806268, ...,  0.1089537 ,
         -0.07971516, -0.14563054],
        [-0.10690635,  0.04080275,  0.04410565, ...,  0.03533241,
          0.01124437, -0.21145652],
        ...,
        [ 0.02110175,  0.15562877,  0.0

**NOTE**: Sometimes, we don't save model file. Instead, we save the architectures and weights.

### Helper functions

In [ ]:
txt = "This movie looks good. But I cannot decide yet"
# Perform punctuation removal, lowering and splitting
words = re.sub(r'[^\w\s]', '', txt.lower()).split()
# If a word doesn't find, return 2
encoded_review = [word_index[word] + 3 if word in word_index else 2 for word in words]
padded_review = sequence.pad_sequences([encoded_review], maxlen=500) # padding = 'pre' (default)
prediction = model.predict(padded_review)[0][0]
["Positive" if prediction > 0.5 else "Negative"]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


['Negative']

In [ ]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i-3, "?") for i in encoded_review])

def preprocess_text(text):
    words = re.sub(r'[^\w\s]', '', txt.lower()).split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

### Prediction

In [ ]:
# Prediction function
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)[0][0]
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    return sentiment, prediction

### Example prediction

In [13]:
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Predicrtion score: {score}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Positive
Predicrtion score: 0.5498790740966797


In [14]:
example_review = "This movie looks good. But I cannot decide yet"
sentiment, score = predict_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Predicrtion score: {score}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Review: This movie looks good. But I cannot decide yet
Sentiment: Negative
Predicrtion score: 0.2748905122280121
